# Pipeline OpenRefine + PartyFacts para conciliacion de partidos (NED)

**Objetivo**: maximizar la cobertura del mapeo `NED_party -> partyfacts_id` usando dos rutas complementarias y comparables.

**Ruta A (recomendada)**: matching directo contra descargas de PartyFacts (Core + Dataset Parties), con limpieza, estandarizacion y criterios reproducibles.

**Ruta B**: reconciliar con Wikidata en OpenRefine y luego enlazar a PartyFacts via el campo `wikipedia` de Core Parties, con fallback por nombre cuando haga falta.

Este notebook prepara insumos para OpenRefine, ejecuta un matching automatico para explorar configuraciones y exporta el mejor resultado junto a una cola de revision manual.

**Mapa del proceso**
1. Cargar eventos electorales base y construir tabla unica de partidos NED.
2. Normalizar nombres (basico y agresivo) y detectar genericos.
3. Cargar PartyFacts y construir tabla de alias.
4. Ruta A: exportar insumos para OpenRefine y probar multiples configuraciones de matching directo.
5. Ruta B: exportar insumos con QID de pais, reconciliar en OpenRefine y enlazar a PartyFacts via Wikipedia.
6. Comparar configuraciones (cobertura y acuerdo con baseline) y elegir la mejor.
7. Exportar el mapeo final y una cola de revision manual.

**Entradas esperadas**
- `data/interim/election_event_sources_base.parquet` (eventos con `party_1_name_raw`, `party_2_name_raw`, `iso3`, `election_year`).
- `data/external/partyfacts_core_parties.csv`.
- `data/external/partyfacts_external_parties.csv` (dataset parties con `partyfacts_id`).

**Salidas principales**
- `data/processed/party_match_openrefine_best.csv`.
- `data/processed/election_event_sources_openrefine_best.parquet`.

**Salidas intermedias (OpenRefine)**
- `data/interim/openrefine_party_match/routeA_ned_parties.csv`.
- `data/interim/openrefine_party_match/routeA_partyfacts_aliases.csv`.
- `data/interim/openrefine_party_match/routeB_ned_parties.csv`.

In [ ]:
from __future__ import annotations

import json
import re
import unicodedata
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from rapidfuzz import process, fuzz

pd.set_option("display.max_columns", 200)

In [ ]:
# Configuracion general
ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
assert (ROOT / "data").exists(), "No se encontro la carpeta data/ en el repo."

paths = {
    "root": ROOT,
    "data": ROOT / "data",
    "interim": ROOT / "data" / "interim",
    "processed": ROOT / "data" / "processed",
    "external": ROOT / "data" / "external",
}

# Ajustes
ALLOW_NETWORK = False  # activar solo si vas a consultar Wikidata desde aqui
RANDOM_SEED = 42

base_path = paths["interim"] / "election_event_sources_base.parquet"
if not base_path.exists():
    raise FileNotFoundError(f"No existe {base_path}. Ejecuta el pipeline base para generarlo.")

events_base = pd.read_parquet(base_path)

# Cargar baseline si existe (para evaluar acuerdo)
baseline_path = paths["processed"] / "election_event_sources.parquet"
baseline = None
if baseline_path.exists():
    baseline = pd.read_parquet(baseline_path)[["record_id", "party_1_id_best", "party_2_id_best"]]

## 1) Tabla unica de partidos NED

Se pasa a formato largo para tratar ganador y runner-up como la misma entidad nominal. Se agregan rangos de anos observados y conteo de eventos para priorizar revision.

In [ ]:
party_long = events_base[[
    "record_id", "iso3", "election_year", "source", "office_type",
    "party_1_name_raw", "party_2_name_raw"
]].melt(
    id_vars=["record_id", "iso3", "election_year", "source", "office_type"],
    value_vars=["party_1_name_raw", "party_2_name_raw"],
    var_name="party_role_raw",
    value_name="party_name_raw",
)

party_long["party_name_raw"] = party_long["party_name_raw"].astype(str).str.strip()
party_long = party_long[party_long["party_name_raw"].notna()]
party_long = party_long[party_long["party_name_raw"] != ""]

party_unique = party_long.groupby(["iso3", "party_name_raw"], as_index=False).agg(
    n_events=("record_id", "nunique"),
    year_first=("election_year", "min"),
    year_last=("election_year", "max"),
)
party_unique["party_recon_key"] = party_unique["iso3"].astype(str) + "||" + party_unique["party_name_raw"].astype(str)

party_unique.head()

## 2) Normalizacion y deteccion de genericos

Se crean dos versiones de limpieza:
- **Basica**: estandariza minusculas, elimina tildes y puntuacion.
- **Agresiva**: elimina terminos comunes (party, movimiento, frente, union, etc.) para mejorar coincidencias con nombres sucios.

Se detectan nombres genericos ("independent", "other", etc.) para excluirlos del matching automatico.

In [ ]:
GENERIC_PARTY_NAMES = {
    "independent", "independiente", "independents", "independent candidate",
    "non partisan", "nonpartisan", "no party", "none", "null",
    "other", "others", "unknown", "n a", "na",
}

STOPWORDS_AGGRESSIVE = {
    "party", "partido", "parti", "parte", "partija", "partie",
    "movement", "movimiento", "moviment", "movimento", "movimentul",
    "front", "frente", "fronte", "frontul",
    "union", "unione", "uniune", "unionen", "uniao",
    "coalition", "coalicion", "coalizione", "koalition", "koalicija",
    "alliance", "alianza", "allianza", "alianse",
    "democratic", "democratica", "democratico", "demokrat",
    "national", "nacional", "nationale", "nazionale",
    "people", "pueblo", "povoa", "peoples", "popor",
    "social", "socialist", "socialista", "sozial",
    "liberal", "liberale", "liberalen",
    "christian", "cristiano", "christdem",
    "republican", "republicano", "republicaine",
    "green", "verde", "verdi",
}


def strip_accents(text: str) -> str:
    if text is None:
        return ""
    text = unicodedata.normalize("NFKD", str(text))
    return "".join(ch for ch in text if not unicodedata.combining(ch))


def normalize_basic(text: str) -> str:
    if text is None:
        return ""
    text = strip_accents(text).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_aggressive(text: str) -> str:
    basic = normalize_basic(text)
    tokens = [t for t in basic.split() if t not in STOPWORDS_AGGRESSIVE]
    if not tokens:
        tokens = basic.split()
    return " ".join(tokens)


def make_acronym(text: str) -> str:
    if text is None:
        return ""
    words = re.findall(r"[A-Za-z]+", str(text))
    if not words:
        return ""
    if len(words) == 1:
        return words[0][:6].upper()
    return "".join(w[0] for w in words).upper()


def is_generic_name(text: str) -> bool:
    norm = normalize_basic(text)
    if norm in GENERIC_PARTY_NAMES:
        return True
    if norm.startswith("independent "):
        return True
    return False


party_unique["party_clean_basic"] = party_unique["party_name_raw"].map(normalize_basic)
party_unique["party_clean_aggressive"] = party_unique["party_name_raw"].map(normalize_aggressive)
party_unique["party_acronym"] = party_unique["party_name_raw"].map(make_acronym)
party_unique["is_generic"] = party_unique["party_name_raw"].map(is_generic_name)

party_unique.head()

## 3) PartyFacts: carga y construccion de alias

Se construye una tabla de alias que combina Core Parties y Dataset Parties, expandiendo variantes de nombres.
La columna `partyfacts_id` es la clave para el merge final.

In [ ]:
pf_core_path = paths["external"] / "partyfacts_core_parties.csv"
pf_external_path = paths["external"] / "partyfacts_external_parties.csv"

if not pf_core_path.exists() or not pf_external_path.exists():
    raise FileNotFoundError("No se encontraron archivos PartyFacts en data/external/")

partyfacts_core = pd.read_csv(pf_core_path, low_memory=False)
partyfacts_external = pd.read_csv(pf_external_path, low_memory=False)

for df in [partyfacts_core, partyfacts_external]:
    df["country"] = df["country"].astype(str).str.upper().str.strip()
    df["partyfacts_id"] = pd.to_numeric(df.get("partyfacts_id"), errors="coerce").astype("Int64")


def explode_aliases(name: str) -> list[str]:
    if pd.isna(name):
        return []
    raw = str(name).strip()
    if not raw:
        return []

    # eliminar prefijos tipo "Idioma:"
    raw = re.sub(r"^[A-Za-z\- ]+:\s*", "", raw)

    parts = re.split(r"\s*[;/|]+\s*", raw)
    out = []
    for part in parts:
        part = part.strip()
        if not part:
            continue
        out.append(part)
        # variante sin parentesis
        no_par = re.sub(r"\s*\([^)]*\)", "", part).strip()
        if no_par and no_par != part:
            out.append(no_par)
        # contenido de parentesis como alias
        for m in re.findall(r"\(([^)]*)\)", part):
            m = m.strip()
            if len(m) > 2 and not m.isdigit():
                out.append(m)
    # deduplicar preservando orden
    seen = set()
    dedup = []
    for a in out:
        if a not in seen:
            seen.add(a)
            dedup.append(a)
    return dedup


def build_alias_table(core: pd.DataFrame, external: pd.DataFrame) -> pd.DataFrame:
    rows = []
    core_cols = ["name_short", "name", "name_english", "name_other"]
    ext_cols = ["name_short", "name", "name_english"]

    for _, r in core.iterrows():
        pid = r.get("partyfacts_id")
        if pd.isna(pid):
            continue
        for col in core_cols:
            for alias in explode_aliases(r.get(col)):
                rows.append({
                    "partyfacts_id": int(pid),
                    "country": r.get("country"),
                    "alias_raw": alias,
                    "alias_source": f"core::{col}",
                    "source_dataset": "core",
                    "year_first": r.get("year_first"),
                    "year_last": r.get("year_last"),
                    "technical": r.get("technical"),
                    "wikipedia": r.get("wikipedia"),
                })

    for _, r in external.iterrows():
        pid = r.get("partyfacts_id")
        if pd.isna(pid):
            continue
        for col in ext_cols:
            for alias in explode_aliases(r.get(col)):
                rows.append({
                    "partyfacts_id": int(pid),
                    "country": r.get("country"),
                    "alias_raw": alias,
                    "alias_source": f"external::{r.get('dataset_key','unknown')}::{col}",
                    "source_dataset": "external",
                    "year_first": r.get("year_first"),
                    "year_last": r.get("year_last"),
                    "technical": r.get("technical"),
                    "wikipedia": None,
                })

    alias_df = pd.DataFrame(rows)
    alias_df["alias_clean_basic"] = alias_df["alias_raw"].map(normalize_basic)
    alias_df["alias_clean_aggressive"] = alias_df["alias_raw"].map(normalize_aggressive)
    alias_df["alias_acronym"] = alias_df["alias_raw"].map(make_acronym)

    alias_df = alias_df[alias_df["alias_clean_basic"].str.len() >= 3].copy()
    alias_df = alias_df.drop_duplicates(subset=["partyfacts_id", "country", "alias_clean_basic"])
    return alias_df


partyfacts_aliases = build_alias_table(partyfacts_core, partyfacts_external)
partyfacts_aliases.head()

## 4) Ruta A: insumos para OpenRefine (matching directo)

Se exportan dos tablas para trabajar en OpenRefine:
- Partidos NED (con versiones limpias y anos observados).
- Alias PartyFacts (Core + External) para comparar y estandarizar por pais.

Esto permite usar Cluster & Edit por pais y hacer matching manual o semiautomatico en la UI.

In [ ]:
openrefine_dir = paths["interim"] / "openrefine_party_match"
openrefine_dir.mkdir(parents=True, exist_ok=True)

routeA_ned_path = openrefine_dir / "routeA_ned_parties.csv"
routeA_alias_path = openrefine_dir / "routeA_partyfacts_aliases.csv"

party_unique.sort_values(["iso3", "party_name_raw"]).to_csv(routeA_ned_path, index=False)
partyfacts_aliases.sort_values(["country", "partyfacts_id"]).to_csv(routeA_alias_path, index=False)

routeA_ned_path, routeA_alias_path

## 5) Ruta A: matching automatico (para evaluar configuraciones)

Se implementa un matching reproducible que simula el criterio de OpenRefine, permitiendo comparar varias configuraciones y elegir la que maximiza cobertura sin degradar demasiado el acuerdo con el baseline.

In [ ]:
@dataclass
class MatchVariant:
    name: str
    clean: str  # "basic" o "aggressive"
    score_cutoff: int
    top_n: int
    year_window: int
    min_gap: int
    acronym_boost: int = 5
    year_boost: int = 5


def year_compat_score(ned_first, ned_last, pf_first, pf_last, window: int, boost: int) -> int:
    try:
        nf = float(ned_first)
        nl = float(ned_last)
    except Exception:
        return 0

    try:
        pf1 = float(pf_first)
        pf2 = float(pf_last)
    except Exception:
        return 0

    ned_mid = (nf + nl) / 2.0
    if pf1 <= ned_mid <= pf2:
        return int(boost)

    dist = min(abs(ned_mid - pf1), abs(ned_mid - pf2))
    if dist <= window:
        return max(1, int(boost // 2))
    return -max(1, int(boost // 2))


def build_alias_index(alias_df: pd.DataFrame, clean_col: str) -> dict:
    out = {}
    for iso, g in alias_df.groupby("country"):
        g = g.reset_index(drop=True).copy()
        out[iso] = {
            "df": g,
            "choices": g[clean_col].fillna("").tolist(),
        }
    return out


def match_routeA_variant(party_df: pd.DataFrame, alias_df: pd.DataFrame, variant: MatchVariant) -> pd.DataFrame:
    clean_col = "alias_clean_basic" if variant.clean == "basic" else "alias_clean_aggressive"
    party_clean_col = "party_clean_basic" if variant.clean == "basic" else "party_clean_aggressive"

    alias_index = build_alias_index(alias_df, clean_col)

    matches = []
    for _, row in party_df.iterrows():
        if row.get("is_generic"):
            continue
        iso = row.get("iso3")
        party_clean = row.get(party_clean_col)
        if not party_clean:
            continue

        if iso not in alias_index:
            continue

        g = alias_index[iso]
        choices = g["choices"]
        if not choices:
            continue

        candidates = process.extract(
            party_clean,
            choices,
            scorer=fuzz.WRatio,
            limit=variant.top_n,
        )

        scored = []
        for _, score_wr, idx in candidates:
            cand = g["df"].iloc[idx]
            score_ts = fuzz.token_set_ratio(party_clean, cand[clean_col])
            score_pr = fuzz.partial_ratio(party_clean, cand[clean_col])
            score = 0.6 * score_wr + 0.2 * score_ts + 0.2 * score_pr

            if row.get("party_acronym") and cand.get("alias_acronym"):
                if row["party_acronym"] == cand["alias_acronym"]:
                    score += variant.acronym_boost

            score += year_compat_score(
                row.get("year_first"),
                row.get("year_last"),
                cand.get("year_first"),
                cand.get("year_last"),
                variant.year_window,
                variant.year_boost,
            )

            scored.append((score, score_wr, cand))

        if not scored:
            continue

        scored.sort(key=lambda x: x[0], reverse=True)
        best = scored[0]
        second = scored[1] if len(scored) > 1 else None

        if best[0] < variant.score_cutoff:
            continue
        if second is not None and (best[0] - second[0]) < variant.min_gap:
            continue

        cand = best[2]
        matches.append({
            "iso3": row.get("iso3"),
            "party_name_raw": row.get("party_name_raw"),
            "partyfacts_id": cand.get("partyfacts_id"),
            "alias_raw": cand.get("alias_raw"),
            "alias_source": cand.get("alias_source"),
            "score": round(best[0], 2),
            "score_wr": round(best[1], 2),
            "variant": variant.name,
            "match_route": "A",
        })

    return pd.DataFrame(matches)


def apply_mapping(events: pd.DataFrame, mapping: pd.DataFrame) -> pd.DataFrame:
    map_df = mapping[["iso3", "party_name_raw", "partyfacts_id", "match_route", "variant"]].drop_duplicates()
    map1 = map_df.rename(columns={
        "party_name_raw": "party_1_name_raw",
        "partyfacts_id": "party_1_id_openrefine",
        "match_route": "party_1_match_route",
        "variant": "party_1_variant",
    })
    map2 = map_df.rename(columns={
        "party_name_raw": "party_2_name_raw",
        "partyfacts_id": "party_2_id_openrefine",
        "match_route": "party_2_match_route",
        "variant": "party_2_variant",
    })

    out = events.merge(map1, on=["iso3", "party_1_name_raw"], how="left")
    out = out.merge(map2, on=["iso3", "party_2_name_raw"], how="left")
    return out


def evaluate_mapping(events: pd.DataFrame, baseline_df: pd.DataFrame | None = None) -> dict:
    winner_matched = int(events["party_1_id_openrefine"].notna().sum())
    runner_matched = int(events["party_2_id_openrefine"].notna().sum())
    n_events = int(len(events))
    coverage_total = (winner_matched + runner_matched) / max(1, 2 * n_events)

    metrics = {
        "winner_matched": winner_matched,
        "runner_matched": runner_matched,
        "n_events": n_events,
        "coverage_total": coverage_total,
    }

    if baseline_df is not None:
        merged = events.merge(baseline_df, on="record_id", how="left")
        mask_w = merged["party_1_id_best"].notna()
        mask_r = merged["party_2_id_best"].notna()
        agree_w = (merged["party_1_id_openrefine"] == merged["party_1_id_best"]) & mask_w
        agree_r = (merged["party_2_id_openrefine"] == merged["party_2_id_best"]) & mask_r
        metrics.update({
            "agree_rate_w": float(agree_w.sum() / max(1, mask_w.sum())),
            "agree_rate_r": float(agree_r.sum() / max(1, mask_r.sum())),
        })

    return metrics


routeA_variants = [
    MatchVariant(name="A_basic_90_y5", clean="basic", score_cutoff=90, top_n=15, year_window=5, min_gap=4),
    MatchVariant(name="A_basic_85_y10", clean="basic", score_cutoff=85, top_n=20, year_window=10, min_gap=3),
    MatchVariant(name="A_basic_80_y20", clean="basic", score_cutoff=80, top_n=25, year_window=20, min_gap=2),
    MatchVariant(name="A_aggr_90_y5", clean="aggressive", score_cutoff=90, top_n=15, year_window=5, min_gap=4),
    MatchVariant(name="A_aggr_85_y10", clean="aggressive", score_cutoff=85, top_n=20, year_window=10, min_gap=3),
    MatchVariant(name="A_aggr_80_y20", clean="aggressive", score_cutoff=80, top_n=25, year_window=20, min_gap=2),
]

routeA_results = {}
routeA_scores = []

for v in routeA_variants:
    match_df = match_routeA_variant(party_unique, partyfacts_aliases, v)
    events_matched = apply_mapping(events_base, match_df)
    metrics = evaluate_mapping(events_matched, baseline)
    metrics.update({"variant": v.name, "route": "A"})

    routeA_results[v.name] = {
        "party_match": match_df,
        "events": events_matched,
        "metrics": metrics,
    }
    routeA_scores.append(metrics)

routeA_scores_df = pd.DataFrame(routeA_scores).sort_values("coverage_total", ascending=False)
routeA_scores_df

## 6) Ruta B: preparacion para reconciliacion con Wikidata

Se crea un archivo de insumo para OpenRefine con el QID de pais (cuando exista) para restringir la reconciliacion.
Si `ALLOW_NETWORK = False`, solo se usa cache local si ya existe.

In [ ]:
import requests

wikidata_iso_cache = paths["interim"] / "wikidata_iso3_qid.csv"


def get_iso3_qid(cache_path: Path, allow_network: bool) -> pd.DataFrame:
    if cache_path.exists():
        return pd.read_csv(cache_path)
    if not allow_network:
        print("No hay cache de ISO3->QID y ALLOW_NETWORK es False. Se omite esta etapa.")
        return pd.DataFrame(columns=["iso3", "qid"])

    sparql = """
    SELECT ?item ?iso3 WHERE {
      ?item wdt:P298 ?iso3 .
    }
    """
    url = "https://query.wikidata.org/sparql"
    r = requests.get(url, params={"format": "json", "query": sparql}, timeout=60)
    r.raise_for_status()
    data = r.json()["results"]["bindings"]
    rows = []
    for b in data:
        qid = b["item"]["value"].split("/")[-1]
        iso3 = b["iso3"]["value"].upper().strip()
        rows.append({"iso3": iso3, "qid": qid})

    out = pd.DataFrame(rows).drop_duplicates(subset=["iso3"])
    out.to_csv(cache_path, index=False)
    return out


iso_qid = get_iso3_qid(wikidata_iso_cache, ALLOW_NETWORK)

routeB_ned_path = openrefine_dir / "routeB_ned_parties.csv"
routeB_ned = party_unique.merge(iso_qid, on="iso3", how="left")
routeB_ned.rename(columns={"qid": "country_qid"}, inplace=True)
routeB_ned.sort_values(["iso3", "party_name_raw"]).to_csv(routeB_ned_path, index=False)

routeB_ned_path

## 7) Ruta B: pasos en OpenRefine (reconciliacion a Wikidata)

Pasos recomendados en la UI de OpenRefine:
1. Importar `routeB_ned_parties.csv`.
2. Usar `party_name_raw` como columna de reconciliacion.
3. Reconcile -> Start reconciling -> servicio Wikidata.
4. Configurar tipo `political party` (Q7278).
5. Agregar constraint de pais usando `country_qid` cuando exista.
6. (Opcional) Usar anos observados para filtrar manualmente casos ambiguos.
7. Agregar columna de `Wikipedia` desde Wikidata (sitelink) para el enlace con PartyFacts.
8. Exportar CSV con columnas: `party_recon_key`, `iso3`, `party_name_raw`, `wikidata_id`, `wikidata_label`, `wikidata_score`, `wikipedia_url`.

Se recomienda exportar dos variantes:
- `wikidata_recon_country.csv`: con constraint de pais.
- `wikidata_recon_nocountry.csv`: sin constraint, solo como respaldo.

In [ ]:
recon_dir = openrefine_dir / "routeB_recon"
recon_dir.mkdir(parents=True, exist_ok=True)

recon_variants = {
    "B_country": recon_dir / "wikidata_recon_country.csv",
    "B_nocountry": recon_dir / "wikidata_recon_nocountry.csv",
}


def load_recon_file(path: Path, variant: str) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()

    df = pd.read_csv(path, low_memory=False)
    required = {"party_recon_key", "iso3", "party_name_raw", "wikidata_id"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Faltan columnas en {path}: {missing}")

    df = df.copy()
    df["wikidata_id"] = df["wikidata_id"].astype(str).str.extract(r"(Q\d+)")[0]
    df["variant"] = variant
    return df


recon_frames = []
for name, p in recon_variants.items():
    recon_frames.append(load_recon_file(p, name))

recon_all = pd.concat(recon_frames, ignore_index=True) if recon_frames else pd.DataFrame()
recon_all.head()

## 8) Ruta B: Wikidata -> PartyFacts via Wikipedia (con fallback por nombre)

La regla principal es: `wikipedia_url` (Wikidata) -> `wikipedia` (PartyFacts Core).
Si falta Wikipedia, se usa el label de Wikidata y se aplica matching por nombre (con thresholds mas estrictos).

In [ ]:
from urllib.parse import unquote


def normalize_wikipedia_url(url: str) -> str:
    if url is None or pd.isna(url):
        return ""
    s = str(url).strip()
    if not s:
        return ""
    s = re.sub(r"^https?://", "", s, flags=re.I)
    s = s.replace("www.", "")
    s = s.split("#")[0].rstrip("/")
    s = unquote(s)
    return s.lower()


partyfacts_core = partyfacts_core.copy()
partyfacts_core["wikipedia_key"] = partyfacts_core["wikipedia"].map(normalize_wikipedia_url)

wiki_map = partyfacts_core[
    partyfacts_core["wikipedia_key"].notna() & (partyfacts_core["wikipedia_key"] != "")
][["partyfacts_id", "country", "wikipedia_key", "year_first", "year_last"]]


def tiebreak_partyfacts(cand: pd.DataFrame, iso3: str, year_first, year_last) -> int | None:
    if cand.empty:
        return None
    c = cand.copy()
    if iso3:
        c["country_match"] = (c["country"] == iso3)
    else:
        c["country_match"] = False

    # prefer matching country
    c = c.sort_values(["country_match"], ascending=False)
    if len(c) == 1:
        return int(c.iloc[0]["partyfacts_id"])

    # prefer temporal overlap
    def overlap_score(r):
        try:
            nf = float(year_first)
            nl = float(year_last)
            pf1 = float(r.get("year_first"))
            pf2 = float(r.get("year_last"))
        except Exception:
            return 0
        mid = (nf + nl) / 2.0
        if pf1 <= mid <= pf2:
            return 1
        return 0

    c["year_overlap"] = c.apply(overlap_score, axis=1)
    c = c.sort_values(["country_match", "year_overlap"], ascending=False)
    return int(c.iloc[0]["partyfacts_id"])


# Ruta B: resolver por Wikipedia
if recon_all.empty:
    routeB_match = pd.DataFrame(columns=["iso3", "party_name_raw", "partyfacts_id", "variant", "match_route"])
else:
    recon_all = recon_all.copy()
    if "wikipedia_url" in recon_all.columns:
        recon_all["wikipedia_key"] = recon_all["wikipedia_url"].map(normalize_wikipedia_url)
    else:
        recon_all["wikipedia_key"] = ""

    matches = []
    for _, r in recon_all.iterrows():
        key = r.get("wikipedia_key")
        iso3 = r.get("iso3")
        if key:
            cand = wiki_map[wiki_map["wikipedia_key"] == key]
            pid = tiebreak_partyfacts(cand, iso3, None, None)
            if pid is not None:
                matches.append({
                    "iso3": iso3,
                    "party_name_raw": r.get("party_name_raw"),
                    "partyfacts_id": pid,
                    "variant": r.get("variant"),
                    "match_route": "B_wikipedia",
                })

    routeB_match = pd.DataFrame(matches)

# Fallback por nombre (solo si hay label)
if not recon_all.empty and "wikidata_label" in recon_all.columns:
    label_df = recon_all[["iso3", "party_name_raw", "wikidata_label", "variant"]].dropna().copy()
    label_df = label_df.drop_duplicates(subset=["iso3", "party_name_raw"])

    label_df["label_for_match"] = label_df["wikidata_label"]
    label_df["party_clean_basic"] = label_df["label_for_match"].map(normalize_basic)
    label_df["party_clean_aggressive"] = label_df["label_for_match"].map(normalize_aggressive)
    label_df["party_acronym"] = label_df["label_for_match"].map(make_acronym)
    label_df["is_generic"] = label_df["label_for_match"].map(is_generic_name)

    # usar un solo variant robusto para fallback
    fb_variant = MatchVariant(
        name="B_label_fallback",
        clean="basic",
        score_cutoff=90,
        top_n=15,
        year_window=10,
        min_gap=4,
    )

    fb_match = match_routeA_variant(
        label_df,
        partyfacts_aliases,
        fb_variant,
    )
    if not fb_match.empty:
        fb_match["match_route"] = "B_label"
        routeB_match = pd.concat([routeB_match, fb_match], ignore_index=True)

routeB_match.head()

## 9) Comparacion de rutas y seleccion de la mejor configuracion

Se comparan variantes de la Ruta A y el resultado de la Ruta B.
Se prioriza cobertura total y, cuando hay baseline, un acuerdo minimo razonable.

In [ ]:
# Evaluar Ruta B si existe
routeB_scores = []
routeB_results = {}

if not routeB_match.empty:
    events_routeB = apply_mapping(events_base, routeB_match)
    metrics_b = evaluate_mapping(events_routeB, baseline)
    metrics_b.update({"variant": "B_wikipedia", "route": "B"})
    routeB_scores.append(metrics_b)
    routeB_results["B_wikipedia"] = {
        "party_match": routeB_match,
        "events": events_routeB,
        "metrics": metrics_b,
    }

scores_df = pd.concat(
    [routeA_scores_df, pd.DataFrame(routeB_scores)],
    ignore_index=True,
).sort_values("coverage_total", ascending=False)

scores_df

In [ ]:
# Seleccionar la mejor configuracion con criterio pragmatismo
MIN_AGREE = 0.90
FALLBACK_AGREE = 0.85

candidates = scores_df.copy()
if baseline is not None and not scores_df.empty:
    candidates = scores_df[
        (scores_df.get("agree_rate_w", 0) >= MIN_AGREE) &
        (scores_df.get("agree_rate_r", 0) >= MIN_AGREE)
    ]
    if candidates.empty:
        candidates = scores_df[
            (scores_df.get("agree_rate_w", 0) >= FALLBACK_AGREE) &
            (scores_df.get("agree_rate_r", 0) >= FALLBACK_AGREE)
        ]
    if candidates.empty:
        candidates = scores_df.copy()

best_row = candidates.sort_values(["coverage_total"], ascending=False).iloc[0]

best_variant = best_row["variant"]
if best_row["route"] == "A":
    best_out = routeA_results[best_variant]
else:
    best_out = routeB_results[best_variant]

# Combinar A + B: usa el mejor A y completa con B (si existe)
combo_match = None
combo_events = None
combo_metrics = None

if not routeB_match.empty:
    bestA_variant = routeA_scores_df.iloc[0]["variant"]
    bestA_match = routeA_results[bestA_variant]["party_match"]

    combo_match = pd.concat([bestA_match, routeB_match], ignore_index=True)
    combo_match = combo_match.drop_duplicates(subset=["iso3", "party_name_raw"], keep="first")

    combo_events = apply_mapping(events_base, combo_match)
    combo_metrics = evaluate_mapping(combo_events, baseline)
    combo_metrics.update({"variant": f"AplusB::{bestA_variant}", "route": "A+B"})

# Seleccion final: si A+B mejora cobertura y no cae acuerdo, usarlo
final_out = best_out
if combo_metrics is not None:
    use_combo = False
    if baseline is None:
        use_combo = combo_metrics["coverage_total"] >= best_out["metrics"]["coverage_total"]
    else:
        agree_w_ok = combo_metrics.get("agree_rate_w", 0) >= FALLBACK_AGREE
        agree_r_ok = combo_metrics.get("agree_rate_r", 0) >= FALLBACK_AGREE
        if agree_w_ok and agree_r_ok and combo_metrics["coverage_total"] >= best_out["metrics"]["coverage_total"]:
            use_combo = True

    if use_combo:
        final_out = {"party_match": combo_match, "events": combo_events, "metrics": combo_metrics}

final_out["metrics"]

In [ ]:
# Export final + cola de revision manual
out_match = paths["processed"] / "party_match_openrefine_best.csv"
out_events = paths["processed"] / "election_event_sources_openrefine_best.parquet"

party_match_best = final_out["party_match"].copy()
events_best = final_out["events"].copy()

party_match_best.to_csv(out_match, index=False)
events_best.to_parquet(out_events, index=False)

# Cola de revision: casos con score cercano o sin match
review_path = paths["processed"] / "party_match_openrefine_review_queue.csv"

# Usar el mejor matching A (si existe) para detectar ambiguos
review_rows = []
for vname, res in routeA_results.items():
    df = res["party_match"].copy()
    df["variant"] = vname
    review_rows.append(df)

review_df = pd.concat(review_rows, ignore_index=True) if review_rows else pd.DataFrame()
review_df = review_df.sort_values(["score"], ascending=True).head(500)
review_df.to_csv(review_path, index=False)

out_match, out_events, review_path

## 10) Checklist de QA

1. Revisar `party_match_openrefine_review_queue.csv` empezando por scores bajos.
2. Verificar casos con `partyfacts_id` tecnico o partidos sin rango de anos en PartyFacts.
3. Validar manualmente partidos con nombres genericos o coaliciones.
4. Documentar cambios manuales y reinyectarlos como alias en OpenRefine para reproducibilidad.